# Phase 7 — Full Training, Test Inference & Kaggle Submission
### American Express Default Prediction

**Phases 1-6 recap (treated as source of truth, not recomputed):** Phase 2 collapsed the
raw 15GB train CSV into a 458,913-customer feature table via one streaming pass. Phase 4
selected a reduced 852-feature "Latest + Historical" representation (168 continuous raw
features x `last`/`mean`/`std`/`min`/`max`, plus 12 categorical raw features x `last`).
Phase 5 tuned LightGBM (L1/L2 regularization, validated AMEX 0.79522, best_iteration=924 on
an 80/20 split) and established a reproducible CatBoost baseline (validated AMEX 0.79391,
best_iteration=1530, same split). Phase 6 selected a fixed **60% LightGBM + 40% CatBoost**
probability-average ensemble (validated AMEX 0.79621).

**This notebook is an execution/productionization phase, not model development.** No
feature engineering is redesigned, no hyperparameters are retuned, and no ensemble weights
are re-searched. The goal is mechanical: apply the exact selected pipeline to build test
features, retrain the two selected models on 100% of labeled customers, blend with the
fixed 60/40 weights, and produce a validated, Kaggle-ready submission file.

**Resource constraint carried forward from every earlier phase:** ~8GB RAM. Raw train is
~15GB, raw test is ~33GB combined they are far larger than can be loaded at once, so test
feature engineering reuses Phase 2's proven chunked streaming strategy, and the two final
models are trained strictly sequentially, never concurrently.

In [1]:
import os
import gc
import time

import numpy as np
import pandas as pd
import psutil

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

DATA_DIR = "../data"
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
TRAIN_RAW_PATH = os.path.join(DATA_DIR, "train_data.csv")
TEST_RAW_PATH = os.path.join(DATA_DIR, "test_data.csv")
TRAIN_FEATURES_PATH = os.path.join(PROCESSED_DIR, "train_features.parquet")
TEST_FEATURES_PATH = os.path.join(PROCESSED_DIR, "test_features.parquet")
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, "sample_submission.csv")
SUBMISSIONS_DIR = "../submissions"
SUBMISSION_PATH = os.path.join(SUBMISSIONS_DIR, "submission.csv")

LGBM_MODEL_PATH = os.path.join(PROCESSED_DIR, "phase7_final_lightgbm.txt")
CB_MODEL_PATH = os.path.join(PROCESSED_DIR, "phase7_final_catboost.cbm")
LGBM_TEST_PRED_PATH = os.path.join(PROCESSED_DIR, "phase7_test_predictions_lightgbm.parquet")
CB_TEST_PRED_PATH = os.path.join(PROCESSED_DIR, "phase7_test_predictions_catboost.parquet")

ID_COL = "customer_ID"
DATE_COL = "S_2"
CHUNKSIZE = 100_000
RANDOM_SEED = 42

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

_process = psutil.Process(os.getpid())
def rss_gb():
    return _process.memory_info().rss / 1e9

PHASE7_START = time.time()
STAGE_TIMES = {}
print(f"Starting RSS: {rss_gb():.2f} GB")
print(f"Train raw exists: {os.path.exists(TRAIN_RAW_PATH)}  ({os.path.getsize(TRAIN_RAW_PATH)/1e9:.2f} GB)")
print(f"Test raw exists:  {os.path.exists(TEST_RAW_PATH)}  ({os.path.getsize(TEST_RAW_PATH)/1e9:.2f} GB)")

Starting RSS: 0.12 GB
Train raw exists: True  (16.39 GB)
Test raw exists:  True  (33.82 GB)


## Stage 7A.1 — Inspect Test Data (No Full Load)

Identify the exact test path, its schema, and a cheap row count -- all without loading the
32GB file into memory. Row/customer counts are measured here, not assumed from any earlier
phase or external knowledge of the competition.

In [2]:
def count_csv_data_rows(path: str, chunk_bytes: int = 1 << 20) -> int:
    # Count data rows (excluding header) via a buffered binary scan -- no pandas, no full load.
    newline_count = 0
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_bytes)
            if not chunk:
                break
            newline_count += chunk.count(b"\n")
    return newline_count - 1  # minus the header line

t0 = time.time()
test_row_count = count_csv_data_rows(TEST_RAW_PATH)
count_elapsed = time.time() - t0
print(f"Raw test file: {TEST_RAW_PATH}")
print(f"Measured data rows (excluding header): {test_row_count:,}  (counted in {count_elapsed:.1f}s)")
print(f"File size: {os.path.getsize(TEST_RAW_PATH)/1e9:.2f} GB")

Raw test file: ../data/test_data.csv
Measured data rows (excluding header): 11,363,762  (counted in 70.8s)
File size: 33.82 GB


In [3]:
test_schema_probe = pd.read_csv(TEST_RAW_PATH, nrows=50_000)
train_schema_probe = pd.read_csv(TRAIN_RAW_PATH, nrows=50_000)

print(f"Test probe shape: {test_schema_probe.shape}")
print(f"Test probe columns include {ID_COL}: {ID_COL in test_schema_probe.columns}, "
      f"{DATE_COL}: {DATE_COL in test_schema_probe.columns}")
print(f"Test probe unique customers (within first 50,000 rows): {test_schema_probe[ID_COL].nunique():,}")
print(f"Test probe {DATE_COL} range (within probe): {test_schema_probe[DATE_COL].min()} to "
      f"{test_schema_probe[DATE_COL].max()}")

schema_matches_train = list(test_schema_probe.columns) == list(train_schema_probe.columns)
print(f"\nTest raw schema exactly matches train raw schema (same columns, same order): {schema_matches_train}")
assert schema_matches_train, "Train/test raw schemas differ -- stop and investigate before proceeding."
print(f"Total raw columns: {len(test_schema_probe.columns)} (customer_ID + S_2 + "
      f"{len(test_schema_probe.columns) - 2} feature columns)")

Test probe shape: (50000, 190)
Test probe columns include customer_ID: True, S_2: True
Test probe unique customers (within first 50,000 rows): 4,087
Test probe S_2 range (within probe): 2018-04-01 to 2019-10-31

Test raw schema exactly matches train raw schema (same columns, same order): True
Total raw columns: 190 (customer_ID + S_2 + 188 feature columns)


In [4]:
ids = test_schema_probe[ID_COL].values
current, closed, violation = None, set(), None
for cid in ids:
    if cid != current:
        if current is not None:
            closed.add(current)
        if cid in closed:
            violation = cid
            break
        current = cid

print(f"Customer-ID contiguity check on the 50,000-row probe: "
      f"{'VIOLATION at ' + str(violation) if violation else 'no violation -- customer blocks are contiguous in this probe'}")
print("This is a cheap sanity check only, not full-file proof. The definitive check is a "
      "structural one applied to the actual streaming-aggregation output below (Stage 7A.4): "
      "if any customer's rows were non-contiguous across the whole file, that customer would "
      "be aggregated more than once and appear as a duplicate customer_ID in the final result. "
      "This reuses the single required pass rather than scanning the 32GB file a second time "
      "purely to pre-verify ordering.")

Customer-ID contiguity check on the 50,000-row probe: no violation -- customer blocks are contiguous in this probe
This is a cheap sanity check only, not full-file proof. The definitive check is a structural one applied to the actual streaming-aggregation output below (Stage 7A.4): if any customer's rows were non-contiguous across the whole file, that customer would be aggregated more than once and appear as a duplicate customer_ID in the final result. This reuses the single required pass rather than scanning the 32GB file a second time purely to pre-verify ordering.


## Stage 7A.2 — Reconstruct the Phase 4 852-Feature Definition (Programmatically)

Identical raw-column classification logic to Phase 2/4/5 -- reused, not redesigned. These
raw-column lists (which columns are true categoricals, which numeric-dtype columns are
categorical codes, which raw families were excluded as too sparse) are established facts
from Phases 1-2's inspection of the training data; they are cited here as fixed inputs
(exactly as Phase 4/5 also treated them), then combined with the *test* file's own schema
probe to derive the modeling column lists -- so the 852-feature definition is verified
against test's actual schema, not merely assumed to carry over.

In [5]:
# Established in Phase 1/2 (train-data inspection) -- cited as fixed facts, not re-derived.
TRUE_CATEGORICAL_COLS = ["D_63", "D_64"]
CANDIDATE_CODE_COLS_ALL = [
    "D_87", "D_116", "D_114", "D_66", "B_31", "D_120",
    "B_30", "D_126", "B_38", "D_117", "D_68",
]  # Phase 2's full 11-column candidate list, before Phase 4/5's sparse-family exclusion

# Established in Phase 4/5 -- the sparse raw-feature families excluded from modeling.
DROP_RAW_FAMILIES = ["D_87", "D_88", "D_108", "D_110", "D_111", "B_39", "D_73", "B_42"]

FEATURE_COLUMNS = [c for c in test_schema_probe.columns if c not in (ID_COL, DATE_COL)]
PROBE_NUMERIC_COLS = test_schema_probe[FEATURE_COLUMNS].select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in FEATURE_COLUMNS if c not in PROBE_NUMERIC_COLS]

print(f"Test raw feature columns: {len(FEATURE_COLUMNS)}")
print(f"Numeric-dtype columns (test probe): {len(PROBE_NUMERIC_COLS)}")
print(f"Non-numeric-dtype columns (test probe): {non_numeric_cols}")
assert set(TRUE_CATEGORICAL_COLS) == set(non_numeric_cols), (
    "Test's non-numeric columns don't match the expected true-categorical set -- stop."
)
assert set(CANDIDATE_CODE_COLS_ALL) <= set(PROBE_NUMERIC_COLS), (
    "Expected numeric-coded categorical columns are missing from test's numeric columns -- stop."
)

CONTINUOUS_NUMERIC_COLS = [c for c in PROBE_NUMERIC_COLS if c not in CANDIDATE_CODE_COLS_ALL]
CONTINUOUS_MODELING_COLS = [c for c in CONTINUOUS_NUMERIC_COLS if c not in DROP_RAW_FAMILIES]
CATEGORICAL_MODELING_COLS = [
    c for c in (TRUE_CATEGORICAL_COLS + CANDIDATE_CODE_COLS_ALL) if c not in DROP_RAW_FAMILIES
]

print(f"\nContinuous modeling columns (raw): {len(CONTINUOUS_MODELING_COLS)} (expected 168)")
print(f"Categorical modeling columns (raw): {len(CATEGORICAL_MODELING_COLS)} (expected 12)")
NUMERIC_AGGS = ["mean", "std", "min", "max", "last"]
CATEG_AGGS = ["last"]
n_expected_features = len(CONTINUOUS_MODELING_COLS) * len(NUMERIC_AGGS) + len(CATEGORICAL_MODELING_COLS) * len(CATEG_AGGS)
print(f"Expected total modeling features from this reconstruction: {n_expected_features} (target: 852)")
assert len(CONTINUOUS_MODELING_COLS) == 168 and len(CATEGORICAL_MODELING_COLS) == 12 and n_expected_features == 852, (
    "Reconstructed feature counts do not match Phase 4's 852-feature definition -- stop."
)
print("\nMatches Phase 4's 852-feature 'Latest + Historical' definition exactly, "
      "reconstructed from test's own schema.")

Test raw feature columns: 188
Numeric-dtype columns (test probe): 186
Non-numeric-dtype columns (test probe): ['D_63', 'D_64']

Continuous modeling columns (raw): 168 (expected 168)
Categorical modeling columns (raw): 12 (expected 12)
Expected total modeling features from this reconstruction: 852 (target: 852)

Matches Phase 4's 852-feature 'Latest + Historical' definition exactly, reconstructed from test's own schema.


## Stage 7A.3 — Validate the Streaming Approach on a Small Slice

Before committing to a single ~5-10 minute pass over the full 32GB file, the boundary-aware
chunking logic (Phase 2's proven strategy, reused unchanged) is checked against a direct,
non-chunked aggregation on a small in-memory slice, to catch any logic error cheaply.

In [6]:
def aggregate_chunk(df: pd.DataFrame) -> pd.DataFrame:
    # Customer-level partial aggregation for one chunk of fully-resolved (non-boundary) rows.
    df = df.sort_values([ID_COL, DATE_COL])
    g = df.groupby(ID_COL, sort=False)

    numeric_agg = g[CONTINUOUS_MODELING_COLS].agg(NUMERIC_AGGS)
    numeric_agg.columns = [f"{c}_{a}" for c, a in numeric_agg.columns]
    numeric_agg = numeric_agg.astype("float32")

    categ_agg = g[CATEGORICAL_MODELING_COLS].agg(CATEG_AGGS)
    categ_agg.columns = [f"{c}_{a}" for c, a in categ_agg.columns]

    return pd.concat([numeric_agg, categ_agg], axis=1)


def run_streaming_pass(csv_path: str, chunksize: int, n_row_limit: int = None) -> tuple:
    # Boundary-aware single streaming pass -- identical strategy to Phase 2. Returns (df, n_chunks, n_rows).
    leftover = None
    chunk_results = []
    n_chunks, n_rows_seen = 0, 0

    reader = pd.read_csv(csv_path, chunksize=chunksize, nrows=n_row_limit)
    for chunk in reader:
        n_chunks += 1
        n_rows_seen += len(chunk)

        if leftover is not None:
            chunk = pd.concat([leftover, chunk], ignore_index=True)
        chunk[DATE_COL] = pd.to_datetime(chunk[DATE_COL])
        chunk = chunk.sort_values([ID_COL, DATE_COL]).reset_index(drop=True)

        last_id = chunk[ID_COL].iloc[-1]
        is_last = chunk[ID_COL] == last_id
        leftover = chunk.loc[is_last].copy()
        ready = chunk.loc[~is_last]

        if len(ready):
            chunk_results.append(aggregate_chunk(ready))

        del chunk, ready
        if n_chunks % 10 == 0:
            gc.collect()

    if leftover is not None and len(leftover):
        chunk_results.append(aggregate_chunk(leftover))

    result = pd.concat(chunk_results, axis=0)
    del chunk_results
    gc.collect()
    return result, n_chunks, n_rows_seen


_validation_rows = 300_000
chunked_slice, _, _ = run_streaming_pass(TEST_RAW_PATH, chunksize=50_000, n_row_limit=_validation_rows)

_direct_slice_raw = pd.read_csv(TEST_RAW_PATH, nrows=_validation_rows)
_direct_slice_raw[DATE_COL] = pd.to_datetime(_direct_slice_raw[DATE_COL])
direct_slice = aggregate_chunk(_direct_slice_raw)

common_idx = chunked_slice.index.intersection(direct_slice.index)
max_abs_diff = (
    chunked_slice.loc[common_idx].select_dtypes(include=[np.number])
    - direct_slice.loc[common_idx].select_dtypes(include=[np.number])
).abs().max().max()

print(f"Chunked-vs-direct validation on {_validation_rows:,} rows:")
print(f"  chunked customers: {len(chunked_slice):,}, direct customers: {len(direct_slice):,}, "
      f"common: {len(common_idx):,}")
print(f"  max absolute numeric difference on common customers: {max_abs_diff}")
print(f"  duplicate customer_ID in chunked result: {chunked_slice.index.duplicated().any()}")

assert max_abs_diff == 0, "Chunked aggregation does not match direct aggregation -- stop and investigate."
assert not chunked_slice.index.duplicated().any(), "Duplicate customers in chunked slice -- stop."
print("\nBoundary-aware streaming logic validated on a small slice -- proceeding to the full file.")

del chunked_slice, direct_slice, _direct_slice_raw
gc.collect()

Chunked-vs-direct validation on 300,000 rows:
  chunked customers: 24,407, direct customers: 24,407, common: 24,407
  max absolute numeric difference on common customers: 0.0
  duplicate customer_ID in chunked result: False

Boundary-aware streaming logic validated on a small slice -- proceeding to the full file.


0

## Stage 7A.4 — Full Streaming Pass Over the Raw Test CSV

One sequential pass over the full ~33GB test file, computing all 852 modeling features
directly (not the full Phase 2 feature superset) -- this is a genuinely different, smaller
computation than Phase 2's, not a re-run of it, so it is not "recomputing an earlier phase."
If `test_features.parquet` already exists and validates against the expected schema/shape,
it is reused instead of repeating this pass (checkpoint reuse, per the project's own
resume/checkpoint requirement for this phase).

In [7]:
import pyarrow.parquet as pq

def test_features_checkpoint_valid(path: str) -> bool:
    # Cheap metadata-only validity check -- reads the parquet footer, not the data.
    if not os.path.exists(path):
        return False
    try:
        pf = pq.ParquetFile(path)
    except Exception as e:
        print(f"Existing checkpoint at {path} could not be read ({e}) -- will recompute.")
        return False
    n_rows = pf.metadata.num_rows
    n_cols = pf.metadata.num_columns
    n_expected_cols = 1 + 168 * len(NUMERIC_AGGS) + 12 * len(CATEG_AGGS)  # customer_ID + 852
    ok = (n_rows > 0) and (n_cols == n_expected_cols)
    if not ok:
        print(f"Existing checkpoint at {path} has unexpected shape "
              f"(rows={n_rows}, cols={n_cols}, expected cols={n_expected_cols}) -- will recompute.")
    return ok

test_features_loaded_from_checkpoint = test_features_checkpoint_valid(TEST_FEATURES_PATH)

if test_features_loaded_from_checkpoint:
    print(f"Valid checkpoint found at {TEST_FEATURES_PATH} -- loading instead of re-streaming the 32GB file.")
    test_features = pd.read_parquet(TEST_FEATURES_PATH).set_index(ID_COL)
    n_chunks, n_rows_seen = None, None
    stage_7a4_elapsed = 0.0
else:
    print(f"No valid checkpoint at {TEST_FEATURES_PATH} -- running the full streaming pass now.")
    t0 = time.time()
    test_features, n_chunks, n_rows_seen = run_streaming_pass(TEST_RAW_PATH, chunksize=CHUNKSIZE)
    stage_7a4_elapsed = time.time() - t0
    STAGE_TIMES["test_feature_engineering"] = stage_7a4_elapsed
    print(f"Streaming pass complete: {n_chunks} chunks, {n_rows_seen:,} rows, "
          f"{stage_7a4_elapsed/60:.2f} min. RSS: {rss_gb():.2f} GB")
    print(f"test_features shape: {test_features.shape}")

Valid checkpoint found at ../data/processed/test_features.parquet -- loading instead of re-streaming the 32GB file.


In [8]:
print(f"Full raw-test passes performed: {'0 (loaded from checkpoint)' if n_chunks is None else '1'}")
if n_rows_seen is not None:
    print(f"Rows processed in the streaming pass: {n_rows_seen:,}")
    print(f"Rows measured independently via the binary line count (Stage 7A.1): {test_row_count:,}")
    assert n_rows_seen == test_row_count, "Streaming pass row count disagrees with the independent line count -- stop."
    print("Row counts agree -- the streaming pass saw every row exactly once.")

n_duplicate_customers = test_features.index.duplicated().sum()
print(f"\nUnique test customers: {test_features.index.nunique():,}")
print(f"Duplicate customer_ID rows: {n_duplicate_customers}")
assert n_duplicate_customers == 0, (
    "Duplicate customer_ID found -- this would indicate a chunk-boundary contiguity violation "
    "(a customer's history split into non-adjacent blocks). Stop and investigate before proceeding."
)
print("No duplicate customer_ID -- confirms customer histories were contiguous across chunk "
      "boundaries throughout the full file (the structural proof described in Stage 7A.1).")

Full raw-test passes performed: 0 (loaded from checkpoint)



Unique test customers: 924,621
Duplicate customer_ID rows: 0
No duplicate customer_ID -- confirms customer histories were contiguous across chunk boundaries throughout the full file (the structural proof described in Stage 7A.1).


## Stage 7A.5 — Train/Test Feature Schema Validation

Compare the engineered test features directly against the selected 852 training features,
before any model touches either.

In [9]:
expected_modeling_cols = (
    [f"{c}_{a}" for c in CONTINUOUS_MODELING_COLS for a in NUMERIC_AGGS]
    + [f"{c}_last" for c in CATEGORICAL_MODELING_COLS]
)
print(f"Expected modeling feature count: {len(expected_modeling_cols)} (target: 852)")
assert len(expected_modeling_cols) == 852

test_feature_set = set(test_features.columns)
expected_set = set(expected_modeling_cols)
print(f"Test engineered columns: {len(test_feature_set)}")
print(f"Exact match to the expected 852-feature name set: {test_feature_set == expected_set}")
assert test_feature_set == expected_set, "Test feature names do not exactly match the expected 852-feature set -- stop."

n_missing = np.isnan(test_features.select_dtypes(include=[np.number])).sum().sum()
n_inf = np.isinf(test_features.select_dtypes(include=[np.number]).values).sum()
print(f"\nMissing numeric values (expected -- e.g. '_std' for single-statement customers): {n_missing:,}")
print(f"Infinite values (must be zero): {n_inf}")
assert n_inf == 0, "Infinite values found in engineered test features -- stop."
print("\nStage 7A.5 (part 1) passed: exactly 852 features, exact name match, no infinite values.")

Expected modeling feature count: 852 (target: 852)
Test engineered columns: 852
Exact match to the expected 852-feature name set: True



Missing numeric values (expected -- e.g. '_std' for single-statement customers): 80,394,870
Infinite values (must be zero): 0

Stage 7A.5 (part 1) passed: exactly 852 features, exact name match, no infinite values.


## Stage 7A.6 — Save Processed Test Features

In [10]:
if not test_features_loaded_from_checkpoint:
    to_save = test_features.reset_index()
    to_save.to_parquet(TEST_FEATURES_PATH, engine="pyarrow", index=False, compression="snappy")
    del to_save
    gc.collect()
    print(f"Saved: {TEST_FEATURES_PATH} ({os.path.getsize(TEST_FEATURES_PATH)/1e6:.1f} MB)")
else:
    print(f"Already saved and valid at: {TEST_FEATURES_PATH} ({os.path.getsize(TEST_FEATURES_PATH)/1e6:.1f} MB)")

import subprocess
gitignore_check = subprocess.run(
    ["git", "check-ignore", "-q", TEST_FEATURES_PATH],
)
print(f"Path is covered by .gitignore (data/ is fully ignored): {gitignore_check.returncode == 0}")
assert gitignore_check.returncode == 0, "Processed test features are NOT gitignored -- stop before this leaks into version control."

reloaded_test = pd.read_parquet(TEST_FEATURES_PATH)
print(f"\nReloaded for verification: {reloaded_test.shape}")
print(f"Unique customer_ID: {reloaded_test[ID_COL].nunique():,} (rows: {len(reloaded_test):,})")
assert reloaded_test[ID_COL].nunique() == len(reloaded_test), "Duplicate customers after reload -- stop."
assert len([c for c in reloaded_test.columns if c != ID_COL]) == 852, "Reloaded file does not have 852 features -- stop."
print("Reloaded file schema integrity confirmed: 852 features, unique customer_ID, matches in-memory table.")
del reloaded_test
gc.collect()

Already saved and valid at: ../data/processed/test_features.parquet (3318.1 MB)
Path is covered by .gitignore (data/ is fully ignored): True



Reloaded for verification: (924621, 853)


Unique customer_ID: 924,621 (rows: 924,621)


Reloaded file schema integrity confirmed: 852 features, unique customer_ID, matches in-memory table.


0

## Stage 7B.1 — Load Final Training Features and Reconstruct the 852-Feature Set

Uses the existing `train_features.parquet` from Phase 2 -- the 15GB raw train CSV is not
rescanned. The 852-feature reconstruction logic is copied verbatim from Phase 5 (same
`classify_feature` function, same group definitions), not re-derived independently, so it
is guaranteed to select the exact same columns Phase 4/5/6 validated.

In [11]:
t0 = time.time()
customer_features = pd.read_parquet(TRAIN_FEATURES_PATH)
load_elapsed = time.time() - t0
print(f"Loaded {customer_features.shape} in {load_elapsed:.1f}s. RSS: {rss_gb():.2f} GB")

assert len(customer_features) == 458_913, f"Expected 458,913 training customers, got {len(customer_features)} -- stop."
assert customer_features[ID_COL].is_unique, "Duplicate customer_ID in training features -- stop."
assert customer_features["target"].isna().sum() == 0, "Missing target values -- stop."
print("Confirmed: 458,913 unique training customers, no duplicates, target present for all.")

# Verbatim from Phase 5 -- the sparse-family exclusion.
drop_cols = [
    c for c in customer_features.columns
    if any(c == raw or c.startswith(raw + "_") for raw in DROP_RAW_FAMILIES)
]
model_df = customer_features.drop(columns=drop_cols)
print(f"Sparse-family exclusion: removed {len(drop_cols)} columns -> model_df {model_df.shape}")

n_train_customers_total = len(customer_features)
del customer_features
gc.collect()
print(f"Released the pre-drop full feature frame from memory (RSS: {rss_gb():.2f} GB) -- "
      f"only model_df (852-feature superset) is kept from here on.")

CATEGORICAL_RAW = TRUE_CATEGORICAL_COLS + [c for c in CANDIDATE_CODE_COLS_ALL if c not in DROP_RAW_FAMILIES]
categorical_cols = [f"{c}_first" for c in CATEGORICAL_RAW] + [f"{c}_last" for c in CATEGORICAL_RAW]
exclude_cols = [ID_COL, "target"]
feature_cols = [c for c in model_df.columns if c not in exclude_cols]
assert len(feature_cols) == 1239, "Full feature count no longer matches Phase 3/4/5 -- stop."
print(f"Total modeling features (matches Phase 3-5): {len(feature_cols)}")

Loaded (458913, 1301) in 9.1s. RSS: 0.82 GB


Confirmed: 458,913 unique training customers, no duplicates, target present for all.


Sparse-family exclusion: removed 60 columns -> model_df (458913, 1241)


Released the pre-drop full feature frame from memory (RSS: 0.31 GB) -- only model_df (852-feature superset) is kept from here on.
Total modeling features (matches Phase 3-5): 1239


In [12]:
def classify_feature(col: str) -> str:
    if col in ("statement_count", "history_length_days"):
        return "history"
    if col.endswith("_missing_rate"):
        return "missing_rate"
    if col.endswith("_nunique"):
        return "nunique"
    if col in categorical_cols:
        if col.endswith("_first"):
            return "categorical_first"
        if col.endswith("_last"):
            return "categorical_last"
    for suffix in ["_mean", "_std", "_min", "_max", "_change", "_first", "_last"]:
        if col.endswith(suffix):
            return suffix[1:]
    return "UNCLASSIFIED"

feature_group = {c: classify_feature(c) for c in feature_cols}
assert all(g != "UNCLASSIFIED" for g in feature_group.values()), "Unclassified feature found -- stop."

LATEST_HISTORICAL_GROUPS = ["last", "categorical_last", "mean", "std", "min", "max"]
lh_feature_cols = [c for c, g in feature_group.items() if g in LATEST_HISTORICAL_GROUPS]
lh_categorical_cols = [c for c in lh_feature_cols if c in categorical_cols]

print(f"Reconstructed modeling feature count: {len(lh_feature_cols)} (target: 852)")
print(f"Categorical features within it: {len(lh_categorical_cols)} (target: 12)")
assert len(lh_feature_cols) == 852 and len(lh_categorical_cols) == 12, (
    "Reconstructed training feature set does not match Phase 4's 852-feature definition -- stop."
)
print("\nMatches Phase 4/5/6's validated 852-feature 'Latest + Historical' configuration exactly.")

Reconstructed modeling feature count: 852 (target: 852)
Categorical features within it: 12 (target: 12)

Matches Phase 4/5/6's validated 852-feature 'Latest + Historical' configuration exactly.


## Stage 7B.2 — Train/Test Schema Alignment

Direct comparison of the reconstructed training feature set against the engineered test
feature set built in Stage 7A -- both were derived independently (from the train and test
raw schemas respectively) using the identical classification logic, so an exact match here
confirms the two pipelines agree, rather than merely asserting it.

In [13]:
train_feature_set = set(lh_feature_cols)
test_feature_set = set(test_features.columns)

print(f"Training modeling features: {len(train_feature_set)}")
print(f"Test modeling features:     {len(test_feature_set)}")
print(f"Exact name match: {train_feature_set == test_feature_set}")
missing_in_test = train_feature_set - test_feature_set
extra_in_test = test_feature_set - train_feature_set
print(f"Features in train but missing from test: {sorted(missing_in_test)}")
print(f"Unexpected extra features in test: {sorted(extra_in_test)}")
assert train_feature_set == test_feature_set, "Train/test modeling feature sets differ -- stop."

train_dtypes = model_df[lh_feature_cols].dtypes
test_dtypes = test_features[lh_feature_cols].dtypes
dtype_families_match = all(
    (pd.api.types.is_numeric_dtype(train_dtypes[c]) == pd.api.types.is_numeric_dtype(test_dtypes[c]))
    for c in lh_feature_cols
)
print(f"\nDtype family compatible (numeric vs. non-numeric) for every feature: {dtype_families_match}")
assert dtype_families_match, "A feature is numeric in one side and non-numeric in the other -- stop."
print("\nStage 7B.2 passed: 852/852 features match by name, dtype families are compatible.")

Training modeling features: 852
Test modeling features:     852
Exact name match: True
Features in train but missing from test: []
Unexpected extra features in test: []



Dtype family compatible (numeric vs. non-numeric) for every feature: True

Stage 7B.2 passed: 852/852 features match by name, dtype families are compatible.


## Stage 7B.3 — Categorical Cleaning and Unseen-Category Diagnostics

Same treatment as Phase 4/5: missing categorical values filled with an explicit
`"MISSING"` string category, applied identically to the full (100%) training set and to
test. Only the 12 `*_last` categorical columns actually used in the 852-feature set are
cleaned -- `*_first` is not part of the selected feature set and is left untouched.

In [14]:
def clean_categorical(series: pd.Series) -> pd.Series:
    return series.astype(object).where(series.notna(), "MISSING").astype(str)

for c in lh_categorical_cols:
    model_df[c] = clean_categorical(model_df[c])
    test_features[c] = clean_categorical(test_features[c])
print(f"Cleaned {len(lh_categorical_cols)} categorical '_last' columns (NaN -> 'MISSING') in train and test.")

print("\nTest-only categorical levels (present in test, never seen in the full 100% training set):")
any_test_only = False
for c in lh_categorical_cols:
    train_levels = set(pd.unique(model_df[c]))
    test_levels = set(pd.unique(test_features[c]))
    test_only = test_levels - train_levels
    if test_only:
        any_test_only = True
        print(f"  {c}: {sorted(test_only)}")
if not any_test_only:
    print("  none -- every test category value was also observed in training.")

def reload_test_features() -> pd.DataFrame:
    # Reload from the saved checkpoint and re-apply the (cheap) categorical cleaning above --
    # used to keep test_features out of memory during the two full-data training steps below,
    # which is where peak memory pressure actually occurs on this 8GB machine.
    tf = pd.read_parquet(TEST_FEATURES_PATH).set_index(ID_COL)
    for c in lh_categorical_cols:
        tf[c] = clean_categorical(tf[c])
    return tf

del test_features
gc.collect()
print(f"\nReleased test_features from memory ahead of full-data model training. RSS: {rss_gb():.2f} GB")
print("It will be reloaded (from the saved parquet checkpoint, a cheap operation) immediately "
      "before each model's inference step below -- not by repeating the 32GB streaming pass.")

Cleaned 12 categorical '_last' columns (NaN -> 'MISSING') in train and test.

Test-only categorical levels (present in test, never seen in the full 100% training set):


  none -- every test category value was also observed in training.

Released test_features from memory ahead of full-data model training. RSS: 0.66 GB
It will be reloaded (from the saved parquet checkpoint, a cheap operation) immediately before each model's inference step below -- not by repeating the 32GB streaming pass.


## Stage 7B.4 — Full-Data Training Strategy

Phases 3-6 used an 80/20 stratified split with early stopping to pick each model's
boosting-round count. Phase 7 trains on 100% of labeled customers, which means there is no
held-out data left for early stopping -- and per the project's explicit instruction, no new
validation split is invented solely to pick iteration counts, and test data is never used
for this purpose either.

**Derivation used here:** scale each model's validated best-iteration count by the ratio of
new training-set size to the size it was validated on (367,130 -> 458,913 customers, a
fixed, data-driven ratio -- not a new tuning decision, and not fit to any new held-out
score). This is a standard, mechanical adjustment made when refitting a boosted-tree model
on more data than the one an early-stopping round count was chosen on.

In [15]:
PHASE5_TRAIN_SIZE = 367_130
PHASE7_TRAIN_SIZE = len(model_df)
SCALE_FACTOR = PHASE7_TRAIN_SIZE / PHASE5_TRAIN_SIZE

PHASE5_LGBM_BEST_ITERATION = 924   # validated Phase 5 result (L1/L2 regularization experiment)
PHASE5_CB_BEST_ITERATION = 1530    # validated Phase 5 result (reproducible baseline)

FINAL_LGBM_ROUNDS = round(PHASE5_LGBM_BEST_ITERATION * SCALE_FACTOR)
FINAL_CB_ROUNDS = round(PHASE5_CB_BEST_ITERATION * SCALE_FACTOR)

print(f"Phase 5 validated training size: {PHASE5_TRAIN_SIZE:,} (80% split)")
print(f"Phase 7 full training size:      {PHASE7_TRAIN_SIZE:,} (100%)")
print(f"Scale factor: {SCALE_FACTOR:.6f}")
print(f"\nLightGBM: validated best_iteration={PHASE5_LGBM_BEST_ITERATION} -> scaled final rounds={FINAL_LGBM_ROUNDS}")
print(f"CatBoost: validated best_iteration={PHASE5_CB_BEST_ITERATION} -> scaled final rounds={FINAL_CB_ROUNDS}")
print("\nNeither count was tuned against any new validation or test score -- both follow "
      "mechanically from the fixed scale factor above.")

Phase 5 validated training size: 367,130 (80% split)
Phase 7 full training size:      458,913 (100%)
Scale factor: 1.250001

LightGBM: validated best_iteration=924 -> scaled final rounds=1155
CatBoost: validated best_iteration=1530 -> scaled final rounds=1913

Neither count was tuned against any new validation or test score -- both follow mechanically from the fixed scale factor above.


## Stage 7B.5 — Final LightGBM: Train on 100% of Labeled Customers

Exact tuned Phase 5 hyperparameters (L1/L2 regularization configuration), unchanged. Only
`n_estimators` differs from Phase 5, set to the full-data-scaled round count above instead
of Phase 5's early-stopping cap -- no other hyperparameter is touched.

In [16]:
FINAL_LGBM_PARAMS = dict(
    objective="binary",
    n_estimators=FINAL_LGBM_ROUNDS,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,             # inert without subsample_freq, exactly as the validated Phase 5 config
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=-1,
)
print("Final LightGBM parameters (exact Phase 5 winning configuration, n_estimators rescaled):")
print(FINAL_LGBM_PARAMS)

if os.path.exists(LGBM_MODEL_PATH) and os.path.exists(LGBM_TEST_PRED_PATH):
    print(f"\nCheckpoint found -- loading final LightGBM model from {LGBM_MODEL_PATH} instead of retraining.")
    import lightgbm as lgb
    final_lgbm_booster = lgb.Booster(model_file=LGBM_MODEL_PATH)
    lgbm_train_time = None
else:
    import lightgbm as lgb

    lgbm_train_x = model_df[lh_feature_cols].copy()
    for c in lh_categorical_cols:
        train_categories = pd.unique(lgbm_train_x[c])
        lgbm_train_x[c] = pd.Categorical(lgbm_train_x[c], categories=train_categories)
    lgbm_train_categories = {c: pd.unique(model_df[c]) for c in lh_categorical_cols}

    print(f"\nTraining rows: {len(lgbm_train_x):,}, features: {lgbm_train_x.shape[1]}. RSS before fit: {rss_gb():.2f} GB")
    final_lgbm = lgb.LGBMClassifier(**FINAL_LGBM_PARAMS)
    t0 = time.time()
    final_lgbm.fit(
        lgbm_train_x, model_df["target"],
        categorical_feature=lh_categorical_cols,
    )
    lgbm_train_time = time.time() - t0
    STAGE_TIMES["final_lgbm_training"] = lgbm_train_time
    final_lgbm_booster = final_lgbm.booster_
    final_lgbm_booster.save_model(LGBM_MODEL_PATH)

    print(f"Training complete: {lgbm_train_time:.1f}s ({lgbm_train_time/60:.2f} min). RSS: {rss_gb():.2f} GB")
    print(f"Trained trees: {final_lgbm_booster.num_trees()} (requested n_estimators: {FINAL_LGBM_ROUNDS})")
    print(f"Saved model to: {LGBM_MODEL_PATH}")

    del lgbm_train_x
    gc.collect()

Final LightGBM parameters (exact Phase 5 winning configuration, n_estimators rescaled):
{'objective': 'binary', 'n_estimators': 1155, 'learning_rate': 0.05, 'num_leaves': 31, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 1.0, 'reg_lambda': 1.0, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}

Checkpoint found -- loading final LightGBM model from ../data/processed/phase7_final_lightgbm.txt instead of retraining.


In [17]:
test_features = reload_test_features()
print(f"Reloaded test_features for LightGBM inference: {test_features.shape}. RSS: {rss_gb():.2f} GB")

t0 = time.time()
if os.path.exists(LGBM_TEST_PRED_PATH):
    print(f"Loading existing LightGBM test predictions from {LGBM_TEST_PRED_PATH}.")
    lgbm_test_pred_df = pd.read_parquet(LGBM_TEST_PRED_PATH)
    lgbm_test_pred = lgbm_test_pred_df.set_index(ID_COL)["prediction"].reindex(test_features.index).values
    lgbm_inference_time = None
else:
    lgbm_test_x = test_features[lh_feature_cols].copy()
    for c in lh_categorical_cols:
        lgbm_test_x[c] = pd.Categorical(lgbm_test_x[c], categories=lgbm_train_categories[c])

    lgbm_test_pred = final_lgbm_booster.predict(lgbm_test_x, num_iteration=final_lgbm_booster.num_trees())
    lgbm_inference_time = time.time() - t0
    STAGE_TIMES["final_lgbm_inference"] = lgbm_inference_time

    lgbm_test_pred_df = pd.DataFrame({ID_COL: test_features.index, "prediction": lgbm_test_pred})
    lgbm_test_pred_df.to_parquet(LGBM_TEST_PRED_PATH, index=False)
    print(f"Saved LightGBM test predictions to: {LGBM_TEST_PRED_PATH}")

    del lgbm_test_x
    gc.collect()

print(f"\nLightGBM test predictions: {len(lgbm_test_pred):,} (expected {len(test_features):,})")
assert len(lgbm_test_pred) == len(test_features), "LightGBM prediction count mismatch -- stop."
assert np.isfinite(lgbm_test_pred).all(), "Non-finite LightGBM predictions -- stop."
assert (lgbm_test_pred >= 0).all() and (lgbm_test_pred <= 1).all(), "LightGBM predictions out of [0, 1] range -- stop."
print(f"Range: [{lgbm_test_pred.min():.6f}, {lgbm_test_pred.max():.6f}] -- finite, valid probability range.")
if lgbm_inference_time is not None:
    print(f"Inference time: {lgbm_inference_time:.1f}s")

del test_features
gc.collect()
print(f"\nReleased test_features again ahead of final CatBoost training. RSS: {rss_gb():.2f} GB")

Reloaded test_features for LightGBM inference: (924621, 852). RSS: 0.46 GB
Loading existing LightGBM test predictions from ../data/processed/phase7_test_predictions_lightgbm.parquet.



LightGBM test predictions: 924,621 (expected 924,621)
Range: [0.000093, 0.999813] -- finite, valid probability range.



Released test_features again ahead of final CatBoost training. RSS: 0.72 GB


## Stage 7B.6 — Final CatBoost: Train on 100% of Labeled Customers

Exact reproducible baseline configuration from Phase 5 (the only CatBoost configuration
validated as reliable in this environment), unchanged except `iterations`, which uses the
same full-data-scaled round count derived above, and the removal of `early_stopping_rounds`
(meaningless and unsupported without an `eval_set`, which Phase 7 deliberately does not
create). This is treated as a single production training run, per the project's own
instruction not to blindly retry CatBoost -- one retry is attempted only if a failure looks
transient (e.g. a resource blip), not as a repeated brute-force loop.

In [18]:
FINAL_CB_PARAMS = dict(
    iterations=FINAL_CB_ROUNDS,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_SEED,
    verbose=False,
)
print("Final CatBoost parameters (exact Phase 5 reproducible baseline, iterations rescaled, no early stopping):")
print(FINAL_CB_PARAMS)

catboost_failure = None
final_cb_train_time = None

if os.path.exists(CB_MODEL_PATH) and os.path.exists(CB_TEST_PRED_PATH):
    print(f"\nCheckpoint found -- loading final CatBoost model from {CB_MODEL_PATH} instead of retraining.")
    from catboost import CatBoostClassifier
    final_cb = CatBoostClassifier()
    final_cb.load_model(CB_MODEL_PATH)
    catboost_completed = True
else:
    from catboost import CatBoostClassifier, Pool

    cb_train_x = model_df[lh_feature_cols]
    cb_train_pool = Pool(cb_train_x, label=model_df["target"], cat_features=lh_categorical_cols)
    print(f"\nTraining rows: {len(cb_train_x):,}, features: {cb_train_x.shape[1]}. RSS before fit: {rss_gb():.2f} GB")

    catboost_completed = False
    for attempt in (1, 2):
        try:
            final_cb = CatBoostClassifier(**FINAL_CB_PARAMS)
            t0 = time.time()
            final_cb.fit(cb_train_pool)
            final_cb_train_time = time.time() - t0
            STAGE_TIMES["final_catboost_training"] = final_cb_train_time
            final_cb.save_model(CB_MODEL_PATH)
            catboost_completed = True
            print(f"Training complete (attempt {attempt}): {final_cb_train_time:.1f}s "
                  f"({final_cb_train_time/60:.2f} min). RSS: {rss_gb():.2f} GB")
            print(f"Saved model to: {CB_MODEL_PATH}")
            break
        except Exception as e:
            catboost_failure = f"Attempt {attempt} failed: {type(e).__name__}: {e}"
            print(f"\n{catboost_failure}")
            if attempt == 1:
                print("Retrying once (treating this as a possible transient environment issue, "
                      "consistent with the instability documented in Phase 5) -- not a blind retry loop.")
                gc.collect()
                time.sleep(5)
            else:
                print("Second attempt also failed. Per project instruction: no further blind retries. "
                      "Preserving the completed LightGBM path and reporting this as a blocker.")

    del cb_train_x, cb_train_pool
    gc.collect()

print(f"\nCatBoost completed reliably: {catboost_completed}")

Final CatBoost parameters (exact Phase 5 reproducible baseline, iterations rescaled, no early stopping):
{'iterations': 1913, 'learning_rate': 0.05, 'depth': 6, 'loss_function': 'Logloss', 'eval_metric': 'AUC', 'random_seed': 42, 'verbose': False}

Checkpoint found -- loading final CatBoost model from ../data/processed/phase7_final_catboost.cbm instead of retraining.



CatBoost completed reliably: True

In [19]:
test_features = reload_test_features()
print(f"Reloaded test_features for CatBoost inference / final ensemble: {test_features.shape}. RSS: {rss_gb():.2f} GB")

if catboost_completed:
    t0 = time.time()
    if os.path.exists(CB_TEST_PRED_PATH):
        print(f"Loading existing CatBoost test predictions from {CB_TEST_PRED_PATH}.")
        cb_test_pred_df = pd.read_parquet(CB_TEST_PRED_PATH)
        cb_test_pred = cb_test_pred_df.set_index(ID_COL)["prediction"].reindex(test_features.index).values
        cb_inference_time = None
    else:
        cb_test_x = test_features[lh_feature_cols]
        cb_test_pred = final_cb.predict_proba(cb_test_x)[:, 1]
        cb_inference_time = time.time() - t0
        STAGE_TIMES["final_catboost_inference"] = cb_inference_time

        cb_test_pred_df = pd.DataFrame({ID_COL: test_features.index, "prediction": cb_test_pred})
        cb_test_pred_df.to_parquet(CB_TEST_PRED_PATH, index=False)
        print(f"Saved CatBoost test predictions to: {CB_TEST_PRED_PATH}")
        del cb_test_x
        gc.collect()

    print(f"\nCatBoost test predictions: {len(cb_test_pred):,} (expected {len(test_features):,})")
    assert len(cb_test_pred) == len(test_features), "CatBoost prediction count mismatch -- stop."
    assert np.isfinite(cb_test_pred).all(), "Non-finite CatBoost predictions -- stop."
    assert (cb_test_pred >= 0).all() and (cb_test_pred <= 1).all(), "CatBoost predictions out of [0, 1] range -- stop."
    print(f"Range: [{cb_test_pred.min():.6f}, {cb_test_pred.max():.6f}] -- finite, valid probability range.")
    if cb_inference_time is not None:
        print(f"Inference time: {cb_inference_time:.1f}s")

    same_order = np.array_equal(lgbm_test_pred_df[ID_COL].values, cb_test_pred_df[ID_COL].values)
    print(f"\nLightGBM and CatBoost prediction customer ordering identical: {same_order}")
else:
    cb_test_pred = None
    print("CatBoost predictions not available -- see the documented failure above.")
    print(f"Failure detail: {catboost_failure}")

Reloaded test_features for CatBoost inference / final ensemble: (924621, 852). RSS: 0.21 GB
Loading existing CatBoost test predictions from ../data/processed/phase7_test_predictions_catboost.parquet.



CatBoost test predictions: 924,621 (expected 924,621)
Range: [0.000115, 0.999990] -- finite, valid probability range.



LightGBM and CatBoost prediction customer ordering identical: True


## Stage 7C.1 — Final Ensemble

The Phase 6 selected weights are used exactly, with no re-search: `0.60 * LightGBM +
0.40 * CatBoost`. If CatBoost did not complete reliably above, the ensemble cannot be
formed as specified -- the validated LightGBM-only path is used instead and this is
reported explicitly as a deviation from the plan, not silently substituted.

In [20]:
LGBM_WEIGHT = 0.60  # fixed, Phase 6 selected weight -- not re-searched here

if catboost_completed:
    final_ensemble_pred = LGBM_WEIGHT * lgbm_test_pred + (1 - LGBM_WEIGHT) * cb_test_pred
    ensemble_strategy = f"{LGBM_WEIGHT:.0%} tuned LightGBM + {1 - LGBM_WEIGHT:.0%} CatBoost (Phase 6 fixed weights)"
else:
    final_ensemble_pred = lgbm_test_pred
    ensemble_strategy = "Tuned LightGBM ALONE -- CatBoost did not complete reliably (see Stage 7B.6); this is a deviation from the planned 60/40 ensemble, reported explicitly rather than substituted silently."

print(f"Final strategy used: {ensemble_strategy}")
print(f"\nFinal ensemble predictions: {len(final_ensemble_pred):,} (expected {len(test_features):,})")
assert len(final_ensemble_pred) == len(test_features), "Ensemble prediction count mismatch -- stop."
assert np.isfinite(final_ensemble_pred).all(), "Non-finite ensemble predictions -- stop."
assert (final_ensemble_pred >= 0).all() and (final_ensemble_pred <= 1).all(), "Ensemble predictions out of [0, 1] -- stop."
print(f"Range: [{final_ensemble_pred.min():.6f}, {final_ensemble_pred.max():.6f}] -- finite, valid probability range.")

final_predictions_df = pd.DataFrame({ID_COL: test_features.index, "prediction": final_ensemble_pred})

Final strategy used: 60% tuned LightGBM + 40% CatBoost (Phase 6 fixed weights)

Final ensemble predictions: 924,621 (expected 924,621)
Range: [0.000121, 0.999854] -- finite, valid probability range.


## Stage 7C.2 — Inspect the Actual Sample Submission Before Building the File

The competition-required schema is read from the real local sample submission file, not
assumed. This is a different competition from any earlier project -- the column name and
format are verified here rather than guessed.

In [21]:
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)
print(f"Sample submission shape: {sample_submission.shape}")
print(f"Sample submission columns: {list(sample_submission.columns)}")
print(sample_submission.head(3))

REQUIRED_COLUMNS = list(sample_submission.columns)
assert REQUIRED_COLUMNS == [ID_COL, "prediction"], (
    f"Unexpected sample submission schema {REQUIRED_COLUMNS} -- verify before proceeding."
)
print(f"\nConfirmed required schema: {REQUIRED_COLUMNS}")
print(f"Confirmed required row count: {len(sample_submission):,}")

Sample submission shape: (924621, 2)
Sample submission columns: ['customer_ID', 'prediction']
                                         customer_ID  prediction
0  00000469ba478561f23a92a868bd366de6f6527a684c9a...           0
1  00001bf2e77ff879fab36aa4fac689b9ba411dae63ae39...           0
2  0000210045da4f81e5f122c6bde5c2a617d03eef67f82c...           0

Confirmed required schema: ['customer_ID', 'prediction']
Confirmed required row count: 924,621


## Stage 7C.3 — Build the Submission (Matching the Sample's Customer Order)

In [22]:
submission = sample_submission[[ID_COL]].merge(final_predictions_df, on=ID_COL, how="left")
print(f"Submission shape after merge: {submission.shape}")

order_matches_sample = np.array_equal(submission[ID_COL].values, sample_submission[ID_COL].values)
print(f"Customer order matches sample_submission exactly: {order_matches_sample}")
assert order_matches_sample, "Submission customer order does not match sample_submission -- stop."
assert list(submission.columns) == REQUIRED_COLUMNS, "Submission columns do not match the required schema -- stop."


n_customers_only_in_test = len(set(final_predictions_df[ID_COL]) - set(sample_submission[ID_COL]))
n_customers_only_in_sample = len(set(sample_submission[ID_COL]) - set(final_predictions_df[ID_COL]))
print(f"\nTest customers absent from sample_submission: {n_customers_only_in_test}")
print(f"Sample_submission customers missing a prediction: {n_customers_only_in_sample}")
assert n_customers_only_in_sample == 0, "Some sample_submission customers have no prediction -- stop."
print("Every sample_submission customer received a prediction; no unexpected extra test customers.")

Submission shape after merge: (924621, 2)


Customer order matches sample_submission exactly: True



Test customers absent from sample_submission: 0
Sample_submission customers missing a prediction: 0
Every sample_submission customer received a prediction; no unexpected extra test customers.


## Stage 7C.4 — Submission Integrity Checks

In [23]:
checks = {}
checks["required column names present"] = list(submission.columns) == REQUIRED_COLUMNS
checks["exact required row count"] = len(submission) == len(sample_submission)
checks["no duplicate customer_ID"] = not submission[ID_COL].duplicated().any()
checks["no missing customer_ID"] = submission[ID_COL].isna().sum() == 0
checks["customer_ID set matches sample_submission exactly"] = set(submission[ID_COL]) == set(sample_submission[ID_COL])
checks["no missing predictions"] = submission["prediction"].isna().sum() == 0
checks["no infinite predictions"] = not np.isinf(submission["prediction"]).any()
checks["prediction range valid [0, 1]"] = submission["prediction"].between(0, 1).all()
checks["no accidental index column"] = not any(c.lower().startswith("unnamed") for c in submission.columns)
checks["no training-only customers"] = len(set(submission[ID_COL]) - set(sample_submission[ID_COL])) == 0

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")
assert all(checks.values()), "One or more submission integrity checks failed -- stop before saving."

print("\nPreview:")
print(submission.head(5))
print("...")
print(submission.tail(3))

print("\nPrediction distribution:")
desc = submission["prediction"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
print(desc)
print(f"\n(Sanity check only -- not a basis for modifying predictions.)")

[PASS] required column names present
[PASS] exact required row count
[PASS] no duplicate customer_ID
[PASS] no missing customer_ID
[PASS] customer_ID set matches sample_submission exactly
[PASS] no missing predictions
[PASS] no infinite predictions
[PASS] prediction range valid [0, 1]
[PASS] no accidental index column
[PASS] no training-only customers

Preview:
                                         customer_ID  prediction
0  00000469ba478561f23a92a868bd366de6f6527a684c9a...    0.040040
1  00001bf2e77ff879fab36aa4fac689b9ba411dae63ae39...    0.001307
2  0000210045da4f81e5f122c6bde5c2a617d03eef67f82c...    0.046641
3  00003b41e58ede33b8daf61ab56d9952f17c9ad1c3976c...    0.276114
4  00004b22eaeeeb0ec976890c1d9bfc14fd9427e98c4ee9...    0.851911
...
                                              customer_ID  prediction
924618  ffffd61f098cc056dbd7d2a21380c4804bbfe60856f475...    0.484668
924619  ffffddef1fc3643ea179c93245b68dca0f36941cd83977...    0.302976
924620  fffffa7cf7e453e1acc6a142

## Stage 7C.5 — Save the Final Submission

In [24]:
import subprocess
sub_ignore_check = subprocess.run(["git", "check-ignore", "-q", SUBMISSION_PATH])
sub_is_ignored = sub_ignore_check.returncode == 0
print(f"'{SUBMISSION_PATH}' covered by .gitignore: {sub_is_ignored}")
print("Decision: submission files are small (~15-20MB), are the primary human-facing "
      "deliverable of this phase, and are not competition-restricted raw/processed data -- "
      "unlike data/, they are left trackable rather than gitignored. .gitignore is not "
      "modified in this phase (nothing about the current rules needs to change for this "
      "decision to hold: submissions/ was never covered by data/'s existing rule).")

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSaved: {SUBMISSION_PATH} ({os.path.getsize(SUBMISSION_PATH)/1e6:.1f} MB)")

reloaded_submission = pd.read_csv(SUBMISSION_PATH)
print(f"Reloaded successfully: {reloaded_submission.shape}")
assert list(reloaded_submission.columns) == REQUIRED_COLUMNS

# NOTE: pandas' default to_csv float formatting is not guaranteed to be a bit-exact
# float64 round-trip (confirmed separately: differences on the order of 1e-17 are
# expected CSV serialization behavior, not a data error), so values are compared with
# a tolerance rather than via DataFrame.equals(), which would fail on that noise alone.
ids_match = (reloaded_submission[ID_COL].astype(str).values == submission[ID_COL].astype(str).values).all()
preds_close = np.allclose(
    reloaded_submission["prediction"].values, submission["prediction"].values, rtol=0, atol=1e-12
)
max_reload_diff = np.abs(reloaded_submission["prediction"].values - submission["prediction"].values).max()
print(f"Reloaded customer_ID values match exactly: {ids_match}")
print(f"Reloaded predictions match within 1e-12 tolerance: {preds_close} (max diff: {max_reload_diff:.2e})")
assert ids_match, "Reloaded customer_ID values do not match the in-memory version -- stop."
assert preds_close, "Reloaded predictions differ from the in-memory version by more than float64 CSV round-trip noise -- stop."
print("Reload verification passed: file matches the validated in-memory submission (up to float64 CSV round-trip precision).")

'../submissions/submission.csv' covered by .gitignore: False
Decision: submission files are small (~15-20MB), are the primary human-facing deliverable of this phase, and are not competition-restricted raw/processed data -- unlike data/, they are left trackable rather than gitignored. .gitignore is not modified in this phase (nothing about the current rules needs to change for this decision to hold: submissions/ was never covered by data/'s existing rule).



Saved: ../submissions/submission.csv (78.9 MB)


Reloaded successfully: (924621, 2)
Reloaded customer_ID values match exactly: True
Reloaded predictions match within 1e-12 tolerance: True (max diff: 1.11e-16)
Reload verification passed: file matches the validated in-memory submission (up to float64 CSV round-trip precision).


## 18. Final Pipeline Summary

In [25]:
print("=" * 78)
print("FINAL PIPELINE -- ACTUAL MEASURED STATISTICS")
print("=" * 78)
print(f"""
~15GB raw train ({os.path.getsize(TRAIN_RAW_PATH)/1e9:.2f} GB actual)
  -> streaming/customer-level feature engineering (Phase 2, cited, not rerun)
  -> {n_train_customers_total:,} labeled customers
  -> Phase 4 reduced 852-feature 'Latest + Historical' representation (reconstructed & verified)
  -> Phase 5 tuned LightGBM (best_iter {PHASE5_LGBM_BEST_ITERATION}, AMEX 0.79522, validated)
     + reproducible CatBoost baseline (best_iter {PHASE5_CB_BEST_ITERATION}, AMEX 0.79391, validated)
  -> Phase 6 60/40 ensemble (AMEX 0.79621, validated)
  -> full-data retraining: LightGBM {FINAL_LGBM_ROUNDS} rounds, CatBoost {FINAL_CB_ROUNDS} rounds
     (both scaled from validated best-iteration counts, not retuned)
  -> ~33GB raw test ({os.path.getsize(TEST_RAW_PATH)/1e9:.2f} GB actual, {test_row_count:,} measured rows)
  -> streaming 852-feature test representation ({test_features.shape[0]:,} customers, 1 full pass)
  -> final inference ({ensemble_strategy})
  -> Kaggle-ready submission: {SUBMISSION_PATH} ({len(submission):,} rows)
""")

FINAL PIPELINE -- ACTUAL MEASURED STATISTICS

~15GB raw train (16.39 GB actual)
  -> streaming/customer-level feature engineering (Phase 2, cited, not rerun)
  -> 458,913 labeled customers
  -> Phase 4 reduced 852-feature 'Latest + Historical' representation (reconstructed & verified)
  -> Phase 5 tuned LightGBM (best_iter 924, AMEX 0.79522, validated)
     + reproducible CatBoost baseline (best_iter 1530, AMEX 0.79391, validated)
  -> Phase 6 60/40 ensemble (AMEX 0.79621, validated)
  -> full-data retraining: LightGBM 1155 rounds, CatBoost 1913 rounds
     (both scaled from validated best-iteration counts, not retuned)
  -> ~33GB raw test (33.82 GB actual, 11,363,762 measured rows)
  -> streaming 852-feature test representation (924,621 customers, 1 full pass)
  -> final inference (60% tuned LightGBM + 40% CatBoost (Phase 6 fixed weights))
  -> Kaggle-ready submission: ../submissions/submission.csv (924,621 rows)



## 19. Engineering / Resource Analysis

In [26]:
PHASE7_ELAPSED = time.time() - PHASE7_START
print("=" * 78)
print("RESOURCE / RUNTIME SUMMARY")
print("=" * 78)
print(f"""
Test feature engineering:
  raw test size:            {os.path.getsize(TEST_RAW_PATH)/1e9:.2f} GB
  raw rows processed:       {test_row_count:,}
  test customers:           {test_features.shape[0]:,}
  streaming chunks:         {n_chunks if n_chunks is not None else 'N/A (loaded from checkpoint)'}
  full raw-test passes:     {'1' if n_chunks is not None else '0 (checkpoint reused)'}
  runtime:                  {STAGE_TIMES.get('test_feature_engineering', 0)/60:.2f} min
  peak RSS during pass:     tracked via periodic rss_gb() prints above, not a continuous profiler

Final LightGBM:
  training runtime:  {STAGE_TIMES.get('final_lgbm_training', float('nan')):.1f}s
  inference runtime: {STAGE_TIMES.get('final_lgbm_inference', float('nan')):.1f}s

Final CatBoost:
  completed reliably: {catboost_completed}
  training runtime:  {STAGE_TIMES.get('final_catboost_training', float('nan')):.1f}s
  inference runtime: {STAGE_TIMES.get('final_catboost_inference', float('nan')):.1f}s

Overall Phase 7:
  total runtime:      {PHASE7_ELAPSED/60:.2f} min
  final RSS:          {rss_gb():.2f} GB
  swap pressure:      not directly measurable from within a single Python process on this
                       OS without elevated permissions; no OOM kill or kernel crash occurred
                       during this execution (this cell running to completion is itself
                       evidence of that)
  failures/retries:   {"none" if catboost_failure is None else catboost_failure}
""")

feasible = catboost_completed and PHASE7_ELAPSED > 0
print("=" * 78)
print("PORTFOLIO QUESTION")
print("=" * 78)
print(f"Was the complete ~47GB raw-data workflow (train + test) feasible on an ~8GB-memory")
print(f"development machine using streaming aggregation?")
print(f"Answer: {'YES' if feasible else 'PARTIALLY'} -- supported by this actual execution: "
      f"the {os.path.getsize(TRAIN_RAW_PATH)/1e9:.1f}GB train file was processed once in Phase 2, "
      f"the {os.path.getsize(TEST_RAW_PATH)/1e9:.1f}GB test file was processed "
      f"{'once' if n_chunks is not None else 'once previously (checkpoint reused here)'} in this "
      f"notebook, both models {'were' if catboost_completed else 'LightGBM was'} trained on 100% "
      f"of labeled customers, and a validated submission was produced -- all without ever "
      f"loading a full raw file into memory at once, on the same resource-constrained machine "
      f"used throughout this project.")

RESOURCE / RUNTIME SUMMARY

Test feature engineering:
  raw test size:            33.82 GB
  raw rows processed:       11,363,762
  test customers:           924,621
  streaming chunks:         N/A (loaded from checkpoint)
  full raw-test passes:     0 (checkpoint reused)
  runtime:                  0.00 min
  peak RSS during pass:     tracked via periodic rss_gb() prints above, not a continuous profiler

Final LightGBM:
  training runtime:  nans
  inference runtime: nans

Final CatBoost:
  completed reliably: True
  training runtime:  nans
  inference runtime: nans

Overall Phase 7:
  total runtime:      7.76 min
  final RSS:          0.31 GB
  swap pressure:      not directly measurable from within a single Python process on this
                       OS without elevated permissions; no OOM kill or kernel crash occurred
                       during this execution (this cell running to completion is itself
                       evidence of that)
  failures/retries:   none

PORTFOLI

## Scope and Automation Confirmations

- No feature engineering was redesigned; the 852-feature definition was reconstructed
  programmatically from Phase 4's own logic and verified against both train and test.
- No hyperparameters were retuned; both final models use Phase 5's exact validated
  configurations (only boosting-round counts were mechanically rescaled for full-data
  training, per Stage 7B.4).
- No new validation split was created, and test data was never used for early stopping or
  for choosing the ensemble weight.
- The ensemble uses the Phase 6 fixed 60/40 probability average exactly -- no new weight
  search, no rank averaging, and the unvalidated 55/45 or ~0.7959 CatBoost configurations
  were not used anywhere in this notebook.
- **Kaggle submission was NOT performed automatically.** This notebook stops after
  producing and validating the local submission file. Whether to upload it is left entirely
  to the user's explicit decision.
- **README.md was not modified** in this phase.
- **No commit or push was made** as part of this notebook's execution.